In [0]:
from pyspark.sql import functions as F

CATALOG = "bucharest_mobility"
BRONZE = "bronze"
SILVER = "silver"

GTFS_TABLES = [
    "agency",
    "stops",
    "routes",
    "calendar",
    "trips",
    "stop_times",
    "shapes",
]

In [0]:
snapshot_id = (
    spark.table(f"{CATALOG}.{BRONZE}.agency")
    .orderBy(F.desc("_ingested_at"))
    .select("_snapshot_id")
    .first()[0]
)

print(f"Processing snapshot: {snapshot_id}")

In [0]:
agency = (
    spark.table(f"{CATALOG}.{BRONZE}.agency")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "agency_id",
        "agency_name",
        "agency_url",
        "agency_timezone",
        "agency_lang",
        "agency_phone",
        "agency_fare_url",
        "agency_email",
        "_snapshot_id",
        "_ingested_at",
    )
)

stops = (
    spark.table(f"{CATALOG}.{BRONZE}.stops")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "stop_id",
        "stop_name",
        "stop_desc",
        F.col("stop_lat").cast("double").alias("stop_lat"),
        F.col("stop_lon").cast("double").alias("stop_lon"),
        F.col("location_type").cast("int").alias("location_type"),
        "platform_code",
        "parent_station",
        "level_id",
        "_snapshot_id",
        "_ingested_at",
    )
)

routes = (
    spark.table(f"{CATALOG}.{BRONZE}.routes")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "route_id",
        "agency_id",
        "route_short_name",
        F.col("route_type").cast("int").alias("route_type"),
        "route_color",
        "route_text_color",
        "route_long_name",
        "_snapshot_id",
        "_ingested_at",
    )
)

In [0]:
calendar = (
    spark.table(f"{CATALOG}.{BRONZE}.calendar")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "service_id",
        *[
            F.col(day).cast("boolean").alias(day)
            for day in [
                "monday",
                "tuesday",
                "wednesday",
                "thursday",
                "friday",
                "saturday",
                "sunday",
            ]
        ],
        F.to_date("start_date", "yyyyMMdd").alias("start_date"),
        F.to_date("end_date", "yyyyMMdd").alias("end_date"),
        "_snapshot_id",
        "_ingested_at",
    )
)

trips = (
    spark.table(f"{CATALOG}.{BRONZE}.trips")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "route_id",
        "service_id",
        "trip_id",
        "trip_headsign",
        "trip_short_name",
        F.col("direction_id").cast("int").alias("direction_id"),
        "block_id",
        "shape_id",
        F.col("wheelchair_accessible")
        .cast("int")
        .alias("wheelchair_accessible"),
        F.col("bikes_allowed").cast("int").alias("bikes_allowed"),
        "_snapshot_id",
        "_ingested_at",
    )
)

In [0]:
shapes = (
    spark.table(f"{CATALOG}.{BRONZE}.shapes")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "shape_id",
        F.col("shape_pt_lat").cast("double").alias("shape_pt_lat"),
        F.col("shape_pt_lon").cast("double").alias("shape_pt_lon"),
        F.col("shape_pt_sequence").cast("int").alias("shape_pt_sequence"),
        "_snapshot_id",
        "_ingested_at",
    )
)

In [0]:

# GTFS times may exceed 24:00:00, so keep the original value
# and convert it to seconds since the start of the service day.

def gtfs_time_to_seconds(column_name):
    parts = F.split(F.col(column_name), ":")

    return (
        parts[0].cast("int") * 3600
        + parts[1].cast("int") * 60
        + parts[2].cast("int")
    )

In [0]:
stop_times = (
    spark.table(f"{CATALOG}.{BRONZE}.stop_times")
    .filter(F.col("_snapshot_id") == snapshot_id)
    .select(
        "trip_id",
        "arrival_time",
        "departure_time",
        gtfs_time_to_seconds("arrival_time").alias("arrival_seconds"),
        gtfs_time_to_seconds("departure_time").alias("departure_seconds"),
        "stop_id",
        F.col("stop_sequence").cast("int").alias("stop_sequence"),
        "stop_headsign",
        F.col("pickup_type").cast("int").alias("pickup_type"),
        F.col("drop_off_type").cast("int").alias("drop_off_type"),
        F.col("shape_dist_traveled")
        .cast("double")
        .alias("shape_dist_traveled"),
        F.col("timepoint").cast("int").alias("timepoint"),
        "_snapshot_id",
        "_ingested_at",
    )
)

In [0]:
SILVER_TABLES = {
    "agency": agency,
    "stops": stops,
    "routes": routes,
    "calendar": calendar,
    "trips": trips,
    "stop_times": stop_times,
    "shapes": shapes,
}

for table_name, df in SILVER_TABLES.items():
    target_table = f"{CATALOG}.{SILVER}.{table_name}"

    if spark.catalog.tableExists(target_table):
        already_processed = (
            spark.table(target_table)
            .filter(F.col("_snapshot_id") == snapshot_id)
            .limit(1)
            .count()
            > 0
        )
    else:
        already_processed = False

    if already_processed:
        print(f"SKIP {table_name}: snapshot already processed")
        continue

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"LOADED {table_name}")

print("Silver transformation complete.")

In [0]:
for table_name in GTFS_TABLES:
    bronze_count = (
        spark.table(f"{CATALOG}.{BRONZE}.{table_name}")
        .filter(F.col("_snapshot_id") == snapshot_id)
        .count()
    )

    silver_count = (
        spark.table(f"{CATALOG}.{SILVER}.{table_name}")
        .filter(F.col("_snapshot_id") == snapshot_id)
        .count()
    )

    assert bronze_count == silver_count, (
        f"{table_name}: Bronze={bronze_count}, Silver={silver_count}"
    )

    print(f"✓ {table_name}: {silver_count:,} rows")

In [0]:
KEYS = {
    "agency": ["agency_id"],
    "stops": ["stop_id"],
    "routes": ["route_id"],
    "calendar": ["service_id"],
    "trips": ["trip_id"],
    "stop_times": ["trip_id", "stop_sequence"],
    "shapes": ["shape_id", "shape_pt_sequence"],
}

for table_name, natural_keys in KEYS.items():
    df = spark.table(f"{CATALOG}.{SILVER}.{table_name}")

    keys = ["_snapshot_id"] + natural_keys

    duplicates = (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0, (
        f"{table_name}: {duplicates} duplicate keys"
    )

    print(f"✓ {table_name}: snapshot grain unique")

In [0]:
CHECKS = [
    ("trips", "route_id", "routes", "route_id"),
    ("stop_times", "trip_id", "trips", "trip_id"),
    ("stop_times", "stop_id", "stops", "stop_id"),
]

for child_table, child_key, parent_table, parent_key in CHECKS:
    child = spark.table(
        f"{CATALOG}.{SILVER}.{child_table}"
    ).alias("child")

    parent = spark.table(
        f"{CATALOG}.{SILVER}.{parent_table}"
    ).alias("parent")

    orphan_count = (
        child.join(
            parent,
            (
                F.col(f"child.{child_key}")
                == F.col(f"parent.{parent_key}")
            )
            & (
                F.col("child._snapshot_id")
                == F.col("parent._snapshot_id")
            ),
            "left_anti",
        )
        .count()
    )

    assert orphan_count == 0, (
        f"{child_table}.{child_key}: "
        f"{orphan_count} orphan rows"
    )

    print(f"✓ {child_table}.{child_key}: valid")